In [46]:
import os
import sys
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pickle
import pandas as pd
from torchvision import datasets, transforms
from torch.utils.data import random_split, DataLoader, ConcatDataset
import math
import scipy.special
import random as rd
import torch.nn.functional as F
import torchvision.models as models
import matplotlib.pyplot as plt
from torchvision.models import VGG16_Weights
from tqdm import tqdm
import pickle
import torch.optim.lr_scheduler as lr_scheduler
from sgr_utils import *

print("GPU Available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0))

GPU Available: True
GPU Name: NVIDIA GeForce RTX 4060


### Loading cifar-10 dataset

In [47]:
# Define transforms for CIFAR-10 dataset
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
]) # imageNet stats by rgb channel

# Load full training set (50,000 samples)
full_train_dataset = datasets.CIFAR10(root="C:/Users/ejeme/Documents/python_repos/CIFAR/cifar_data", train=True, transform=transform, download=True)
# Load official test set (10,000 samples)
official_test_dataset = datasets.CIFAR10(root="C:/Users/ejeme/Documents/python_repos/CIFAR/cifar_data", train=False, transform=transform, download=True)

# Keep only classes 0 (airplane) and 1 (automobile)
filtered_train_dataset = filter_classes(full_train_dataset, [0, 1])
filtered_test_dataset = filter_classes(official_test_dataset, [0, 1])

In [48]:
len(filtered_train_dataset)

10000

In [49]:
len(filtered_test_dataset)

2000

In [50]:
train_size = 5000
extra_test_size = len(filtered_train_dataset) - train_size
# Use a fixed seed for reproducibility
generator = torch.Generator().manual_seed(42)
train_dataset, extra_test_dataset = random_split(filtered_train_dataset, [train_size, extra_test_size], generator=generator)
# Concatenate the 20k extra test samples with the official test set
combined_test_dataset = ConcatDataset([extra_test_dataset, filtered_test_dataset])
# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=6, shuffle=True)
test_loader = DataLoader(combined_test_dataset, batch_size=6, shuffle=False)

## Training small CNN for cifar-2 classification task

In [35]:
# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class SmallCNN(nn.Module):
    def __init__(self):
        super(SmallCNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.adaptive_pool = nn.AdaptiveAvgPool2d((4, 4))  # Force feature map to 4x4
        self.fc1 = nn.Linear(32 * 4 * 4, 64)
        self.fc2 = nn.Linear(64, 2)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.adaptive_pool(x)             # Output shape: [B, 32, 4, 4]
        x = x.view(x.size(0), -1)             # Flatten safely: [B, 512]
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x



# Initialize the model
cnn = SmallCNN().to(device)

In [36]:
# Define Loss, Optimizer and lr scheduler
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(cnn.parameters(), lr=1e-4)
scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.1, patience=0)

# Training Loop
num_epochs = 5
for epoch in range(num_epochs):
    cnn.train()
    running_loss = 0
    correct = 0
    total = 0

    print('TRAINING EPOCH', epoch)
    for images, labels in tqdm(train_loader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = cnn(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
            
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}")
    print(f"Train Accuracy: {100 * correct / total:.2f}%")

    # Evaluate cnn at the end of current epoch
    cnn.eval()
    correct = 0
    total = 0
    print('TESTING')
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = cnn(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    print(f"Validation Accuracy: {100 * correct / total:.2f}%")
    scheduler.step(correct)

    torch.save(cnn.state_dict(), "C:/Users/ejeme/Documents/python_repos/CIFAR/cnn_cifar_binary_epoch"+str(epoch)+".pth")

TRAINING EPOCH 0


100%|██████████| 834/834 [00:07<00:00, 115.20it/s]


Epoch [1/5], Loss: 0.5389
Train Accuracy: 73.58%
TESTING
Validation Accuracy: 78.34%
TRAINING EPOCH 1


100%|██████████| 834/834 [00:19<00:00, 41.71it/s] 


Epoch [2/5], Loss: 0.4164
Train Accuracy: 80.94%
TESTING
Validation Accuracy: 81.77%
TRAINING EPOCH 2


...

Epoch [3/5], Loss: 0.3750
Train Accuracy: 82.84%
TESTING
Validation Accuracy: 83.40%
TRAINING EPOCH 3


100%|██████████| 834/834 [00:07<00:00, 117.26it/s]


Epoch [4/5], Loss: 0.3556
Train Accuracy: 84.20%
TESTING
Validation Accuracy: 83.57%
TRAINING EPOCH 4


100%|██████████| 834/834 [00:07<00:00, 117.61it/s]


Epoch [5/5], Loss: 0.3393
Train Accuracy: 84.70%
TESTING
Validation Accuracy: 84.47%


Epoch 4 with 84% test accuracy is enough (model still has to commit mistakes otherwise SGR is not useful)

### Retrieving Softmax Response, Predicted class and True class for all samples in train and in test sets

In [51]:
# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Load trained cnn
cnn = SmallCNN().to(device)
checkpoint = torch.load("C:/Users/ejeme/Documents/python_repos/CIFAR/cnn_cifar_binary_epoch4.pth")
cnn.load_state_dict(checkpoint)

<All keys matched successfully>

In [54]:
sgr_dico = prepare_sgr_dico(test_loader, model = cnn, device = device, T = 1)
sgr_set = pd.DataFrame(sgr_dico)
pickle.dump(sgr_set, open('sgr_set2','wb'))

100%|██████████| 1167/1167 [00:07<00:00, 158.61it/s]


In [55]:
sgr_set.sort_values('SR')

,y_true,y_pred,SR
2116,1.0,1.0,0.500055
26,0.0,0.0,0.500869
2240,0.0,1.0,0.501013
2010,1.0,0.0,0.501031
6686,1.0,0.0,0.501098
...,...,...,...
514,0.0,0.0,0.999956
6229,0.0,0.0,0.999965
2036,0.0,0.0,0.999995
44,0.0,0.0,0.999997
